# Step 4-B: 리플레이 버퍼 & Double DQN 가이드

## 1. 리플레이 버퍼 (Replay Buffer) 설계

### A. 왜 경험을 모아서 섞어야 하는가?
게임 데이터는 시간 순서대로 강하게 연결(Correlated)되어 있습니다. 
순서대로 학습하면 신경망이 방금 겪은 국면만 편향되게 학습하다가 이전 지식을 까먹는 **재앙적 망각(Catastrophic Forgetting)**이 일어납니다.
버퍼에 수만 개의 경험을 모아두고 **무작위(Uniform Random) 미니배치**로 섞어 뽑으면 데이터 간의 상관관계가 깨져 학습이 안정됩니다.

### B. `sample` 메서드의 반환 텐서 형상:
- `states`: `torch.FloatTensor` of shape `(B, 6, 9, 9)`
- `actions`: `torch.LongTensor` of shape `(B,)`
- `rewards`: `torch.FloatTensor` of shape `(B,)`
- `next_states`: `torch.FloatTensor` of shape `(B, 6, 9, 9)`
- `dones`: `torch.FloatTensor` of shape `(B,)` (0.0 또는 1.0)
- `next_masks`: `torch.BoolTensor` of shape `(B, 140)`

---

## 2. 더블 DQN (Double DQN) 손실 함수

### A. 과대평가(Overestimation) 방지 원리
일반 DQN은 `Target = r + gamma * max_a Q_target(s', a)`를 씁니다.
`max` 연산자는 노이즈 때문에 Q값을 실제보다 과대평가하는 치명적인 결함이 있습니다.
**더블 DQN**은 이를 해결하기 위해 두 네트워크의 역할을 분리합니다:
1. **행동 선택(Action Selection)**: `policy_net`이 다음 상태에서 최고 점수의 행동 $a^*$을 선택
   $$a^* = \arg\max_{a} Q_{\text{policy}}(s', a, \text{mask}')$$
2. **가치 평가(Value Evaluation)**: `target_net`이 그 행동 $a^*$의 실제 Q값을 산출
   $$Q_{\text{eval}} = Q_{\text{target}}(s', a^*)$$

### B. 2인 제로섬(Zero-Sum) 게임의 타깃 수식:
다음 상태 $s'$는 **상대방의 턴**입니다!
따라서 상대방의 최대 가치($Q_{\text{eval}}$)는 나에게는 **손실(-)**이 됩니다.
$$\text{Target} = \text{reward} - \gamma \cdot Q_{\text{eval}} \cdot (1 - \text{done})$$

### C. 파이토치 텐서 연산 팁:
1. **현재 Q값 추출**:
   - `policy_net(states)`는 `(B, 140)`입니다.
   - 내가 실제로 취했던 `actions` (B,)에 해당하는 Q값만 뽑아야 합니다:
   - `current_q = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)`
2. **다음 행동 $a^*$ 선택**:
   - `next_actions = policy_net(next_states, mask=next_masks).argmax(dim=1)`
3. **타깃망에서 $a^*$의 가치 추출**:
   - `with torch.no_grad():` 블록 안에서 타깃망 순전파 (기울기 차단!)
   - `next_q = target_net(next_states).gather(1, next_actions.unsqueeze(1)).squeeze(1)`
4. **손실 계산**:
   - `F.smooth_l1_loss(current_q, target)`